In [ ]:
import torch
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

N_ENVS =  20  # roughly the number of physical CPU cores you can spare
torch.set_num_threads(1)  # avoid CPU oversubscription (workers + torch threads)

def make_train_env(n_envs=N_ENVS):
    # Each worker gets its own Monitor so episode stats are tracked per env
    return SubprocVecEnv(
        [lambda: Monitor(make_env()) for _ in range(n_envs)],
        start_method="fork",   # Linux; avoids pickling issues in Jupyter
    )


In [ ]:
import optuna
import numpy as np
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CallbackList, StopTrainingOnMaxEpisodes


# Semillas de validacion comunes a todos los trials, distintas de entrenamiento.
EVAL_SEEDS = [seed + 1000 + i for i in range(EVAL_EPISODES)]

# Diccionario para almacenar informacion sobre el mejor modelo entrenado
best_training_info = {}

# Definir la funcion para el/los entrenamiento/s
def objective(trial):

    # Hiperparametros a optimizar
    learning_rate = trial.suggest_float(
        "learning_rate", 5e-4, 5e-4, log=True   # Si quieres modificar el learning rate, recuerda cambiar el rango de valores aqui. Actualmente esta fijo en 5e-4
    )

    # Resto de hiperparametros a optimizar
    # ...


    train_env = None
    eval_env = None

    try:
        # train_env = Monitor(make_env())
        train_env = make_train_env()
        eval_env = DummyVecEnv([lambda: Monitor(make_env())])

        # Crear agente
        agent = PPO('MlpPolicy',
                    train_env, # No modificar
                    verbose=0, # No modificar
                    seed=seed, # No modificar

                    # Hiperparametros a optimizar
                    learning_rate=learning_rate,
                    # Resto de hiperparametros
                    )


        # Instanciar los callbacks para este trial
        stop_training_callback = StopTrainingOnMaxEpisodes(max_episodes=TRAIN_EPISODES, verbose=1) # Limita el numero de episodios al indicado
        text_progress_bar_callback = TextProgressBarCallback() # Muestra una barra de progreso de texto durante el entrenamiento

        # Instancia aqui las clases definidas en la celda de callbacks
        # custom_callback = MiCallback(...)


        # Lista de callbacks. Incluye aqui los callbacks que quieras aplicar.
        callbacks = CallbackList([
            stop_training_callback,
            text_progress_bar_callback,
            # custom_callback,
            # ...
        ])


        # Entrenamiento del agente
        agent.learn(total_timesteps=TRAIN_EPISODES * TIMESTEPS_PER_EPISODE,
                    callback=callbacks,
                    )


        # Evaluacion de la politica del agente
        episode_rewards = []
        for eval_seed in EVAL_SEEDS:
            eval_env.seed(eval_seed)
            rewards, _ = evaluate_policy(
                agent,
                eval_env,
                n_eval_episodes=1,
                deterministic=True,
                return_episode_rewards=True,
            )
            episode_rewards.extend(rewards)
        mean_reward = float(np.mean(episode_rewards))

        # Si el agente es el mejor hasta ahora, se guarda el agente junto con informacion relevante
        completed_trials = trial.study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
        
        if not completed_trials or mean_reward > trial.study.best_value:
            agent.save(f"agentes/{EXPERIMENT_NAME}/best_model.zip")

            # best_training_info.update({
            #     "rewards": np.asarray(train_env.get_episode_rewards()).copy(),
            #     "lengths": np.asarray(train_env.get_episode_lengths()).copy(),
            #     "trial_number": trial.number,
            #     "validation_mean_reward": mean_reward,
            # })
            rewards = np.concatenate(train_env.env_method("get_episode_rewards"))
            lengths = np.concatenate(train_env.env_method("get_episode_lengths"))

            best_training_info.update({
                "rewards": rewards.copy(),
                "lengths": lengths.copy(),
                "trial_number": trial.number,
                "validation_mean_reward": mean_reward,
            })

        # Se devuelve la recompensa media obtenida durante la evaluacion del agente entrenado
        return mean_reward

    finally:
        if eval_env is not None:
            eval_env.close()
        if train_env is not None:
            train_env.close()


In [ ]:
best_training_info
# plot x rewards y lengths
import matplotlib.pyplot as plt

plt.plot(best_training_info["rewards"][70:100], best_training_info["lengths"][70:100])
plt.xlabel("Rewards")
plt.ylabel("Lengths")
plt.title("Training Performance")
plt.show()

In [ ]:
def make_env():
    new_env = gym.make(TRAIN_ENV_NAME, render_mode='rgb_array',
                        config={
                            "lanes_count": 4,         
                            "vehicles_count": 50,
                            "target_speed": 40, 
                            "reward_speed_range": [20, 30],
                            "vehicles_params": {
                                "speed": 35,                 # Velocidad base superior para los coches de la IA
                            }
                        },
                    )

    new_env = aplicar_wrappers(new_env)
    
    return new_env

env = make_env()

observation, info = env.reset(seed=seed)

# Mostrar el estado inicial
plt.imshow(env.render())
plt.show()

print("Observaciones:", env.observation_space)
print("Acciones:", env.action_space)
env.close()

In [ ]:
# Descomposicion de la recompensa por episodio en highway-v0 (el entorno del test)
# Recompensa por paso: r = base + velocidad + carril + colision, con
#   base = -c/span, velocidad = a*s/span, carril = b*l/span, colision = c*col/span   (span = a + b - c)
import pandas as pd

analysis_agent = PPO.load(f"agentes/{EXPERIMENT_NAME}/best_model.zip")
analysis_env = aplicar_wrappers(gym.make("highway-v0"))  # mismos wrappers que en entrenamiento y test
cfg = analysis_env.unwrapped.config
a, b, c = cfg["high_speed_reward"], cfg["right_lane_reward"], cfg["collision_reward"]
span = (a + b) - c  # rango de la normalizacion: [c, a + b] -> [0, 1]
MAX_STEPS = cfg["duration"] * cfg["policy_frequency"]  # 40 pasos en highway-v0

filas = []
for eval_seed in EVAL_SEEDS:
    state, info = analysis_env.reset(seed=eval_seed)
    terminated = truncated = False
    suma = dict(base=0.0, velocidad=0.0, carril=0.0, colision=0.0)
    total, pasos = 0.0, 0
    while not (terminated or truncated):
        action, _ = analysis_agent.predict(state, deterministic=True)
        state, reward, terminated, truncated, info = analysis_env.step(action)
        comp, on_road = info["rewards"], info["rewards"]["on_road_reward"]
        suma["base"]      += -c / span * on_road
        suma["velocidad"] += a * comp["high_speed_reward"] / span * on_road
        suma["carril"]    += b * comp["right_lane_reward"] / span * on_road
        suma["colision"]  += c * comp["collision_reward"] / span * on_road
        total += reward
        pasos += 1
    # Recompensa que se habria seguido acumulando si no hubiera chocado (al ritmo medio del episodio)
    perdida = (MAX_STEPS - pasos) * total / pasos if info["crashed"] else 0.0
    filas.append(dict(semilla=eval_seed, recompensa=total, pasos=pasos, choque=info["crashed"],
                      **{f"%{k}": 100 * v / total for k, v in suma.items()},
                      perdida_por_choque=perdida))
analysis_env.close()

df_recompensa = pd.DataFrame(filas).round(2)
print(df_recompensa.to_string(index=False))
print("\nMedia de los episodios:")
print(df_recompensa.drop(columns=["semilla"]).mean().round(2).to_string())
df_recompensa.to_csv(f"agentes/{EXPERIMENT_NAME}/descomposicion_recompensa.csv", index=False)